In [ ]:
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity

# If needed: %pip install gensim scikit-learn


# Use article text from the 20 Newsgroups dataset
news = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes")
)
documents = news.data

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

tokenized_documents = [tokenize(text) for text in documents]
tokenized_documents = [tokens for tokens in tokenized_documents if tokens]

# sg=0 selects CBOW
cbow_model = Word2Vec(
    sentences=tokenized_documents,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=10
)

def average_embedding(tokens):
    vectors = [cbow_model.wv[word] for word in tokens if word in cbow_model.wv]
    return np.mean(vectors, axis=0) if vectors else None

# Represent each article by the average of its word embeddings
document_vectors = []
valid_document_indices = []

for index, text in enumerate(documents):
    vector = average_embedding(tokenize(text))
    if vector is not None:
        document_vectors.append(vector)
        valid_document_indices.append(index)

document_vectors = np.vstack(document_vectors)

# Replace this with a user's search or input text
user_input = "space exploration, astronomy, and planetary science"
query_vector = average_embedding(tokenize(user_input))

if query_vector is None:
    print("The input contains no words present in the trained vocabulary.")
else:
    similarities = cosine_similarity(
        query_vector.reshape(1, -1), document_vectors
    )[0]
    top_positions = np.argsort(similarities)[::-1][:5]

    for position in top_positions:
        document_index = valid_document_indices[position]
        print(f"\nSimilarity: {similarities[position]:.3f}")
        print(documents[document_index][:700].replace("\n", " "))